# Start here: what does a grouping tell us?

**The object we evaluate is a grouping—not a gene or an embedding.** Follow the same 560 synthetic observations through four questions. Labels and representations are fixed before prediction; these are teaching assignments, not validated cell types.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import seaborn as sns
import gauge_sc as gauge

adata = gauge.datasets.audit_example()
group = adata.obs["group"].astype(str)
adata.obs["coarse"] = group.map({
    "M0": "M01", "M1": "M01", "M2": "M23", "M3": "M23",
    "Q0": "Q", "Q1": "Q", "Q2": "Q",
})
adata.obs["resolved"] = group.where(~group.str.startswith("Q"), "Q")
adata.obs["fine"] = group
result = gauge.evaluate(
    adata, groupby=["coarse", "resolved", "fine"],
    views={
        "molecular": {"obsm": "X_pca"},
        "qc": {"obs": ["total_counts", "n_genes_by_counts", "pct_counts_mt"],
               "log1p": ["total_counts", "n_genes_by_counts"]},
    },
)


## 1. Which grouping are we asking about?

In [ ]:
# Same observation order in every row; horizontal position is not an embedding.
colors = {"M01": "#1f77b4", "M23": "#279e68", "Q": "#aa40fc",
          "M0": "#1f77b4", "M1": "#ff7f0e", "M2": "#279e68", "M3": "#d62728",
          "Q0": "#aa40fc", "Q1": "#8c564b", "Q2": "#e377c2"}
with sns.plotting_context("paper"), plt.rc_context({"text.color": "black", "axes.labelcolor": "black"}):
    fig, ax = plt.subplots(figsize=(7.2, 2.4), layout="constrained")
    for y, name in enumerate(["coarse", "resolved", "fine"]):
        labels = adata.obs[name].to_numpy()
        starts = np.r_[0, np.flatnonzero(labels[1:] != labels[:-1]) + 1]
        ends = np.r_[starts[1:], len(labels)]
        for start, end in zip(starts, ends):
            label = labels[start]
            ax.add_patch(Rectangle((start, y - .3), end - start, .6,
                                   facecolor=colors[label], edgecolor="white", linewidth=1))
            ax.text((start + end) / 2, y, label, ha="center", va="center",
                    color="black", bbox={"facecolor":"white", "edgecolor":"none", "alpha":.85, "pad":1})
    ax.set(xlim=(0, adata.n_obs), ylim=(2.55, -.55), xticks=[], yticks=[0,1,2],
           yticklabels=["Coarse · 3 groups", "Resolved · 5 groups", "Fine · 7 groups"],
           xlabel="Same observations, same order")
    ax.tick_params(left=False, bottom=False, labelcolor="black")
    for spine in ax.spines.values():spine.set_visible(False)
    plt.show()

We change assignments, not cells. M0–M3 have distinct molecular programs; Q0–Q2 share one. The finest grouping adds distinctions that expression may not support.

## 2. Why isn't prediction accuracy enough?

In [ ]:
predictions = result.predictions.query("view == 'molecular'")
accuracy = predictions.assign(correct=predictions.group == predictions.predicted_group).groupby("groupby").correct.mean()
comparison = result.table.query("view == 'molecular'").set_index("groupby")[["n_groups", "itca"]].copy()
comparison.insert(1, "held_out_accuracy", accuracy)
comparison.rename(columns={"n_groups": "Groups", "held_out_accuracy": "Held-out accuracy", "itca": "ITCA (nats)"}).round(3)

Coarse and resolved labels are both predicted accurately, but the resolved grouping carries more distinguishable label information. **ITCA combines information with held-out prediction support.**

<details><summary>The one formula</summary>

Each group contributes **p × log(1/p) × recall**. Sum these contributions to obtain ITCA. `p` is its population fraction; recall is how well its labels are predicted out of fold. Rare labels carry more information per observation, but population weighting means a rarer group does not always contribute more.

</details>

## 3. How much extra detail is worth keeping?

In [ ]:
selection = gauge.select(result, view="molecular", population_ratio=[0.01, 0.1])
selection.path[["population_ratio", "groupby", "n_groups", "itca", "penalty", "itca_plus"]].round(3)

**ITCA+ subtracts a cost for additional groups.** A mild cost retains five groups; a stronger cost retains three. The population ratio calibrates that cost—it does not filter small groups. These are explicit example choices, not universal defaults.

## 4. Supported by which view?

In [ ]:
audit = gauge.audit(result, groupby="fine", reference="molecular", contrast="qc")
fig = gauge.pl.audit(audit, panel="contributions", annotate=["Q0", "Q1", "Q2"])

The same Q groups receive more support from QC than expression. **Support depends on the view; it is not proof of biological truth or artifact.** QC covariates are independently simulated, not calculated from the expression matrix.

Now choose a task: [select a grouping](00_select_grouping.ipynb), [compare groupings](03_compare_groupings.ipynb), or [audit a view](01_view_specific_audit.ipynb). [All applications](README.md).

## Work with your agent

[Install the skill and choose an interface](../docs/agents.md). Then try:

> Use $gauge to walk me through this example: the same cells, different groupings and views. Explain what a predictable distinction means, why a high score depends on the view, and how ITCA+ changes the granularity decision. Keep the explanation tied to the tables and figures.

These tutorial settings are an example; adapt them to your question and data.